In [3]:
# -*- coding: utf-8 -*-
import os
import json
import time
import numpy as np
import tensorflow as tf

# ==========================================
# KONFIGURASI PATH
# ==========================================
# Arahkan ke file TFLite asli MCU-Quake
TFLITE_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/mcquake_ori_file/Code & Figure demo/Pre-trained model/MCU-Quake 5-20/lite_model.tflite'
# Path ke direktori embedding KDE Indonesia Anda
EMB_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon/output/indonesia_domain_embeddings_3c_le"
EMBEDDING_DIM = 32

def main():
    print("="*60)
    print("BUILD & PROFILING: MCU-QUAKE (KOMPONEN Z) DENGAN EMBEDDING INDONESIA")
    print("="*60)

    # 1. Analisis Model TFLite Asli (Feature Extractor)
    if os.path.exists(TFLITE_PATH):
        flash_size_kb = os.path.getsize(TFLITE_PATH) / 1024.0
        
        interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
        interpreter.allocate_tensors()
        
        tensor_details = interpreter.get_tensor_details()
        ram_bytes = sum([np.prod(t['shape']) * np.dtype(t['dtype']).itemsize for t in tensor_details if t['shape'] is not None])
        
        # Uji latensi inferensi dummy
        input_details = interpreter.get_input_details()
        input_shape = input_details[0]['shape']
        dummy_input = np.random.randn(*input_shape).astype(np.float32)
        
        interpreter.set_tensor(input_details[0]['index'], dummy_input)
        start_time = time.time()
        for _ in range(100):
            interpreter.set_tensor(input_details[0]['index'], dummy_input)
            interpreter.invoke()
        latency_ms = ((time.time() - start_time) / 100) * 1000

        print(f"\n[A] Analisis Feature Extractor (TFLite):")
        print(f"    - Ukuran Flash (ROM) : {flash_size_kb:.2f} KB")
        print(f"    - Estimasi RAM       : {ram_bytes / 1024:.2f} KB")
        print(f"    - Latensi Rata-rata  : {latency_ms:.2f} ms (di PC)")
    else:
        print(f"\n[!] File TFLite tidak ditemukan di: {TFLITE_PATH}")
        print("    Pastikan Anda mengarahkannya ke file .tflite bawaan repositori.")

    # 2. Analisis Beban Memori KDE Indonesia (Khusus Komponen Z)
    print(f"\n[B] Analisis Ruang Probabilitas (KDE Komponen Z):")
    file_path = os.path.join(EMB_DIR, "Embedding data, Z.json")
    
    total_vectors = 0
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            data = json.load(f)
        n_count = len(data.get("noise", []))
        le_count = len(data.get("le", []))
        total_vectors = n_count + le_count
        print(f"    - Komponen Z: {total_vectors} vektor ({n_count} Noise, {le_count} LE)")
    else:
        print(f"    - Komponen Z: File JSON tidak ditemukan di direktori!")

    # Kalkulasi memori KDE dengan asumsi kuantisasi INT8 (1 Byte per parameter)
    kde_ram_kb = (total_vectors * EMBEDDING_DIM * 1) / 1024.0

    print(f"\n    > Total Vektor Laten Komponen Z : {total_vectors} titik")
    print(f"    > Beban Memori Tambahan (INT8)  : {kde_ram_kb:.2f} KB")

    print("\n" + "="*60)
    print("RINGKASAN KELAYAKAN TINYML (DEPLOYMENT SUMMARY)")
    print("="*60)
    print(f"Model dengan Komponen Z tunggal siap di-deploy ke ESP32/STM32.")
    print(f"Footprint memori sistem terintegrasi menjadi sangat ramping (~{kde_ram_kb:.2f} KB untuk KDE).")
    print("="*60)

if __name__ == "__main__":
    main()

BUILD & PROFILING: MCU-QUAKE (KOMPONEN Z) DENGAN EMBEDDING INDONESIA

[A] Analisis Feature Extractor (TFLite):
    - Ukuran Flash (ROM) : 16.16 KB
    - Estimasi RAM       : 84.38 KB
    - Latensi Rata-rata  : 0.01 ms (di PC)

[B] Analisis Ruang Probabilitas (KDE Komponen Z):
    - Komponen Z: 10318 vektor (5159 Noise, 5159 LE)

    > Total Vektor Laten Komponen Z : 10318 titik
    > Beban Memori Tambahan (INT8)  : 322.44 KB

RINGKASAN KELAYAKAN TINYML (DEPLOYMENT SUMMARY)
Model dengan Komponen Z tunggal siap di-deploy ke ESP32/STM32.
Footprint memori sistem terintegrasi menjadi sangat ramping (~322.44 KB untuk KDE).


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [9]:
# -*- coding: utf-8 -*-
import os
import tensorflow as tf

# ==========================================
# KONFIGURASI PATH
# ==========================================
# Path model Keras hasil retraining
KERAS_MODEL_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/frozen_extractor_indonesia_Z.keras"

# Path tujuan untuk model TFLite
TFLITE_OUTPUT_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/frozen_extractor_indonesia_Z.tflite"

def convert_keras_to_tflite():
    print("="*60)
    print("KONVERSI KERAS KE TFLITE")
    print("="*60)
    print(f"[INFO] Memuat model Keras dari:\n       {KERAS_MODEL_PATH}")
    
    if not os.path.exists(KERAS_MODEL_PATH):
        print("\n❌ Gagal: File .keras tidak ditemukan! Pastikan proses pelatihan sudah selesai.")
        return

    # 1. Memuat model Keras
    model = tf.keras.models.load_model(KERAS_MODEL_PATH, compile=False)

    # 2. Inisialisasi TFLite Converter
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    # [OPSIONAL] Kuantisasi untuk memperkecil ukuran model (umum untuk TinyML)
    # Jika ESP32-S3 Anda kehabisan memori, hilangkan tanda komentar pada baris di bawah ini:
    # converter.optimizations = [tf.lite.Optimize.DEFAULT]
    
    # 3. Proses Konversi
    print("[INFO] Memulai proses konversi ke format TFLite...")
    tflite_model = converter.convert()

    # 4. Menyimpan file TFLite
    with open(TFLITE_OUTPUT_PATH, "wb") as f:
        f.write(tflite_model)
        
    print(f"\n✅ Ekspor Berhasil!")
    print(f"   Lokasi Tersimpan : {TFLITE_OUTPUT_PATH}")
    print(f"   Ukuran TFLite    : {len(tflite_model) / 1024:.2f} KB")
    print("="*60)

if __name__ == "__main__":
    convert_keras_to_tflite()

KONVERSI KERAS KE TFLITE
[INFO] Memuat model Keras dari:
       /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/frozen_extractor_indonesia_Z.keras
[INFO] Memulai proses konversi ke format TFLite...
INFO:tensorflow:Assets written to: /var/folders/lt/2mkl6ry53ll9fdk2br6skfgw0000gn/T/tmpmgu7nw40/assets


INFO:tensorflow:Assets written to: /var/folders/lt/2mkl6ry53ll9fdk2br6skfgw0000gn/T/tmpmgu7nw40/assets



✅ Ekspor Berhasil!
   Lokasi Tersimpan : /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/frozen_extractor_indonesia_Z.tflite
   Ukuran TFLite    : 46.49 KB


2026-08-09 16:53:33.000824: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-08-09 16:53:33.000836: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-08-09 16:53:33.000970: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/lt/2mkl6ry53ll9fdk2br6skfgw0000gn/T/tmpmgu7nw40
2026-08-09 16:53:33.002097: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-08-09 16:53:33.002102: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/lt/2mkl6ry53ll9fdk2br6skfgw0000gn/T/tmpmgu7nw40
2026-08-09 16:53:33.004995: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-08-09 16:53:33.025376: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/lt/2mkl6ry53ll9fdk2br6skfgw0000gn/T/tmpmgu7nw40
2026-08-

In [15]:
# -*- coding: utf-8 -*-
import os
import json

# ==========================================
# KONFIGURASI PATH
# ==========================================
TFLITE_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/frozen_extractor_indonesia_Z.tflite'

# Ubah EMB_DIR hanya sampai nama foldernya saja
EMB_DIR = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_eval_03"

# Direktori Output untuk file C++
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models'

def ensure_dir():
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

def export_tflite_to_c_array():
    print("\n[1] Mengekspor Model TFLite ke C-Array...")
    if not os.path.exists(TFLITE_PATH):
        print(f"  ❌ File tidak ditemukan: {TFLITE_PATH}")
        return

    with open(TFLITE_PATH, "rb") as f:
        tflite_data = f.read()

    hex_array = [f"0x{b:02x}" for b in tflite_data]
    c_code = "#ifndef MCU_QUAKE_MODEL_H\n#define MCU_QUAKE_MODEL_H\n\n"
    c_code += "// File TFLite Feature Extractor MCU-Quake\n"
    c_code += f"const unsigned int mcu_quake_model_len = {len(tflite_data)};\n"
    c_code += "const unsigned char mcu_quake_model[] = {\n    "
    
    for i in range(0, len(hex_array), 12):
        c_code += ", ".join(hex_array[i:i+12]) + ",\n    "
    c_code = c_code.rstrip(",\n    ") + "\n};\n\n#endif // MCU_QUAKE_MODEL_H"

    out_path = os.path.join(OUTPUT_DIR, "mcu_quake_model.h")
    with open(out_path, "w") as f:
        f.write(c_code)
    print(f"  ✅ Tersimpan: {out_path} ({len(tflite_data)/1024:.2f} KB)")

def export_kde_z_to_c_array():
    print("\n[2] Mengekspor Matriks KDE Komponen Z ke C-Array...")
    
    # Gabungkan folder EMB_DIR dengan nama file json-nya
    json_path = os.path.join(EMB_DIR, "Embedding data, Z.json")
    
    if not os.path.exists(json_path):
        print(f"  ❌ File tidak ditemukan: {json_path}")
        return

    with open(json_path, "r") as f:
        data = json.load(f)

    noise_vecs = data.get("noise", [])
    le_vecs = data.get("le", [])

    if len(noise_vecs) == 0:
        print("  ❌ Data vektor kosong.")
        return

    # DETEKSI OTOMATIS: Mengecek apakah ini array 1D atau 2D
    first_item = noise_vecs[0]
    is_2d = isinstance(first_item, (list, tuple))
    
    c_code = "#ifndef KDE_Z_VECTORS_H\n#define KDE_Z_VECTORS_H\n\n"
    c_code += "// Ruang Probabilitas Lokal Indonesia (Komponen Z)\n"
    
    # --- FUNGSI PEMBANTU UNTUK FORMATTING ---
    def format_array(name, vectors):
        code = f"const int NUM_{name.upper()} = {len(vectors)};\n"
        
        if is_2d: # Jika Nested Array (Misal: [[0.1, 0.2], [0.3, 0.4]])
            dim = len(vectors[0])
            code += f"const float kde_{name.lower()}_vectors[][{dim}] = {{\n"
            for vec in vectors:
                formatted_vec = ", ".join([f"{val:.6f}" for val in vec])
                code += f"    {{{formatted_vec}}},\n"
            code = code.rstrip(",\n") + "\n};\n\n"
            
        else: # Jika Flat Array 1D (Misal: [0.1, 0.2, 0.3, 0.4])
            code += f"const float kde_{name.lower()}_vectors[] = {{\n    "
            formatted_vecs = [f"{val:.6f}" for val in vectors]
            for i in range(0, len(formatted_vecs), 12):
                code += ", ".join(formatted_vecs[i:i+12]) + ",\n    "
            code = code.rstrip(",\n    ") + "\n};\n\n"
            
        return code

    # Menjalankan formatting untuk derau (Noise) dan gempa (LE)
    c_code += format_array("NOISE", noise_vecs)
    c_code += format_array("LE", le_vecs)
    c_code += "#endif // KDE_Z_VECTORS_H"

    out_path = os.path.join(OUTPUT_DIR, "kde_z_vectors.h")
    with open(out_path, "w") as f:
        f.write(c_code)
    
    bentuk_array = "2D (Nested Array)" if is_2d else "1D (Flat Array)"
    print(f"  ✅ Tersimpan: {out_path} (Terdeteksi Format: {bentuk_array})")

def main():
    print("="*60)
    print("GENERATE C-ARRAY UNTUK ESP32-S3 DEPLOYMENT")
    print("="*60)
    ensure_dir()
    export_tflite_to_c_array()
    export_kde_z_to_c_array()
    print("="*60)
    print("Proses selesai. Pindahkan file .h di dalam folder 'esp32_deployment_files'")
    print("ke dalam folder proyek C++ / Arduino IDE Anda.")
    print("="*60)

if __name__ == "__main__":
    main()

GENERATE C-ARRAY UNTUK ESP32-S3 DEPLOYMENT

[1] Mengekspor Model TFLite ke C-Array...
  ✅ Tersimpan: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/mcu_quake_model.h (46.49 KB)

[2] Mengekspor Matriks KDE Komponen Z ke C-Array...
  ✅ Tersimpan: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/output_models/kde_z_vectors.h (Terdeteksi Format: 2D (Nested Array))
Proses selesai. Pindahkan file .h di dalam folder 'esp32_deployment_files'
ke dalam folder proyek C++ / Arduino IDE Anda.


In [17]:
# -*- coding: utf-8 -*-
import json
import os

# ==========================================
# KONFIGURASI PATH
# ==========================================
DATA_JSON_PATH = "/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia/indonesia_test_data.json"
OUTPUT_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "test_samples.h")

def main():
    # Memastikan folder output tersedia
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        print(f"📁 Membuat folder baru: {OUTPUT_DIR}")

    print("Membaca dataset pengujian...")
    with open(DATA_JSON_PATH, "r") as f:
        test_data = json.load(f)
    
    keys = list(test_data.keys())
    
    # Tentukan jumlah sampel multi-data yang ingin diekstrak (misal: 5 sampel)
    num_samples = min(5, len(keys)) # Mencegah IndexError jika total data kurang dari 5
    
    c_code = "#ifndef TEST_SAMPLES_H\n#define TEST_SAMPLES_H\n\n"
    c_code += "// Kumpulan Sampel Pengujian Multi-Data untuk ESP32-S3\n\n"
    
    for i in range(num_samples):
        k = keys[i]
        sample_eq = test_data[k]["Z"][:700]          # Sinyal Gempa (700 sampel)
        sample_noise = test_data[k]["Z_noise"][-700:] # Sinyal Derau (700 sampel)
        
        # Format Array C untuk Gempa
        c_code += f"const float test_wave_earthquake_{i+1}[700] = {{\n    "
        c_code += ", ".join([f"{val:.6f}" for val in sample_eq])
        c_code += "\n}};\n\n"
        
        # Format Array C untuk Derau
        c_code += f"const float test_wave_noise_{i+1}[700] = {{\n    "
        c_code += ", ".join([f"{val:.6f}" for val in sample_noise])
        c_code += "\n}};\n\n"
    
    c_code += "#endif // TEST_SAMPLES_H"
    
    with open(OUTPUT_FILE, "w") as f:
        f.write(c_code)
        
    print(f"✅ Berhasil mengekstrak {num_samples} pasang gelombang uji ke: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Membaca dataset pengujian...
✅ Berhasil mengekstrak 5 pasang gelombang uji ke: /Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/retraining_mcu_q_indonesia/data_indonesia/test_samples.h
